<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature Vector Assembly & Pipeline StrategyTo build an honest feature matrix for our Refresh / Content Opportunity Scoring lane, we construct numerical indicators from pre-decision search signals (clicks, impressions, position, word count, engagement).  Categorical features like content_type and main_intent are frequency-encoded or one-hot encoded, missing numeric values are imputed with domain-safe fill strategies (e.g., median or zero fill), and explicit checks guarantee that no lookahead target features are passed.  

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 Code: Build Feature Vector Pipeline
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Environment setup & data loading
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 2. Define Ground Truth Target (Decline Flag)
if "target" not in df.columns:
    df["target"] = (df["trend_direction"] == "down").astype(int)

# 3. Engineer Feature Vector
# Numerical Features available AT decision time
num_features = [
    "impressions_90d", "clicks_90d", "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d", "avg_position", "ctr",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "engagement_rate", "scroll_rate", "cpc", "search_volume", "competition"
]

# Keep existing numerical columns
num_features = [c for c in num_features if c in df.columns]

X_num = df[num_features].copy()

# Fill missing numerical values with median
for col in X_num.columns:
    X_num[col] = X_num[col].fillna(X_num[col].median())

# Handle Categorical Features (One-Hot Encoding)
cat_cols = [c for c in ["content_type", "main_intent"] if c in df.columns]
if cat_cols:
    X_cat = pd.get_dummies(df[cat_cols], drop_first=True)
    X = pd.concat([X_num, X_cat], axis=1)
else:
    X = X_num.copy()

print(f"Feature Vector Built: {X.shape[0]:,} rows x {X.shape[1]} features")
print("\nSample Feature Vector Columns:")
print(list(X.columns[:10]))

Feature Vector Built: 30,000 rows x 22 features

Sample Feature Vector Columns:
['impressions_90d', 'clicks_90d', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'avg_position', 'ctr', 'word_count', 'char_count']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature Contract & Data DictionaryEvery feature in the vector is audited to confirm its meaning, missingness strategy, and temporal availability:impressions_90d / clicks_90d: Total organic exposure/clicks over the 90-day observation window. Missing: filled with 0.0. Available: Before prediction timestamp.  impressions_last_30d / impressions_prev_30d: 30-day velocity comparison metrics. Missing: median imputed. Available: Before prediction timestamp.  avg_position & ctr: Organic ranking placement and click-through rate. Missing: avg_position median imputed, ctr filled with 0.0. Available: Before prediction timestamp.  word_count & content_age_days: Page structural metadata and age since publishing/updating. Missing: median imputed. Available: Before prediction timestamp.  content_type & main_intent: Categorical attributes encoded via dummy variables. Missing: categorized as UNKNOWN. Available: Static content metadata.  

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 Code: Feature Missingness and Availability Audit
feature_summary = []

for col in X.columns:
    raw_missing = df[col].isnull().sum() if col in df.columns else 0
    missing_pct = (raw_missing / len(df)) * 100
    feature_summary.append({
        "Feature Name": col,
        "Raw Null Count": raw_missing,
        "Null Pct": round(missing_pct, 2),
        "Data Type": str(X[col].dtype),
        "Available Pre-Decision": "YES"
    })

feature_audit_df = pd.DataFrame(feature_summary)
print("=== Feature Contract & Availability Audit ===")
print(feature_audit_df.head(10).to_string(index=False))

=== Feature Contract & Availability Audit ===
        Feature Name  Raw Null Count  Null Pct Data Type Available Pre-Decision
     impressions_90d               0      0.00     int64                    YES
          clicks_90d               0      0.00     int64                    YES
impressions_last_30d               0      0.00     int64                    YES
impressions_prev_30d               0      0.00     int64                    YES
     clicks_last_30d               0      0.00     int64                    YES
     clicks_prev_30d               0      0.00     int64                    YES
        avg_position               0      0.00   float64                    YES
                 ctr               0      0.00   float64                    YES
          word_count            7699     25.66   float64                    YES
          char_count            7699     25.66   float64                    YES


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Adversarial Leakage InspectionWe attack our candidate feature matrix by checking for direct target-derived proxies and future lookahead windows:  trend_pct Attack: trend_pct measures exact percentage traffic change. Because target is defined as trend_direction == 'down' (derived directly from trend_pct < 0), including trend_pct gives a artificial 100% precision score.  trend_direction Attack: Categorical representation of the target.  Automated Safeguard: We run a strict correlation test and name-filter check across all features in $X$ to guarantee no target proxy leaks through.  

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 Code: Leakage Guard Test & Correlation Audit
forbidden_leak_terms = ["target", "trend_pct", "trend_direction", "future", "flyrank_flag"]

detected_leaks = [
    col for col in X.columns
    if any(leak_term in col.lower() for leak_term in forbidden_leak_terms)
]

print("=== Automated Leakage Guard Audit ===")
if len(detected_leaks) == 0:
    print("✅ PASS: No forbidden target leakage terms found in feature vector X.")
else:
    print(f"❌ FAIL: Potential leakage columns detected: {detected_leaks}")

# Calculate feature correlations with target
correlations = X.apply(lambda col: col.corr(df["target"])).abs().sort_values(ascending=False)

print("\n=== Top 5 Feature Correlations with Target (Post-Audit) ===")
print(correlations.head(5).round(4).to_string())

# Ensure max correlation is within realistic, non-leaking bounds (< 0.90)
assert correlations.max() < 0.90, "Leakage Alert: A feature shows suspiciously high correlation with the target!"
print("✅ PASS: All feature correlations are within honest predictive limits.")

=== Automated Leakage Guard Audit ===
✅ PASS: No forbidden target leakage terms found in feature vector X.

=== Top 5 Feature Correlations with Target (Post-Audit) ===
content_age_days                0.1639
content_type_feedly article     0.1405
content_type_keyword article    0.1183
impressions_last_30d            0.0940
word_count                      0.0843
✅ PASS: All feature correlations are within honest predictive limits.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Explicit Exclusions ListThe following fields were explicitly excluded from the model feature vector:trend_pct: Excluded because it directly computes the target label outcome (direct target leakage).  trend_direction: Excluded because it is the string representation of the ground-truth target.  id / content_id / content_hash_id: Excluded because high-cardinality primary keys lead to memorization rather than learning generalizable signals.  client_id / domain_hash: Excluded from input features (retained solely as a grouping column in validation splits) to prevent the model from memorizing specific client site authority levels.  Future Window Performance Flags: Any post-observation metrics occurring outside the decision window were removed to prevent lookahead bias

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 Code: Summary Table of Refused/Excluded Columns
excluded_columns = [
    {"Column Name": "trend_pct", "Exclusion Reason": "Direct target derivation (100% data leakage)"},
    {"Column Name": "trend_direction", "Exclusion Reason": "Raw label string (direct target leakage)"},
    {"Column Name": "content_id / content_hash_id", "Exclusion Reason": "High cardinality row ID (causes memorization)"},
    {"Column Name": "client_id / domain_hash", "Exclusion Reason": "Client identity key (reserved exclusively for grouped split validation)"},
    {"Column Name": "future_* metrics", "Exclusion Reason": "Post-decision window lookahead bias"}
]

excluded_df = pd.DataFrame(excluded_columns)
print("=== Excluded Fields & Refusal Reasons ===")
print(excluded_df.to_string(index=False))

=== Excluded Fields & Refusal Reasons ===
                 Column Name                                                        Exclusion Reason
                   trend_pct                            Direct target derivation (100% data leakage)
             trend_direction                                Raw label string (direct target leakage)
content_id / content_hash_id                           High cardinality row ID (causes memorization)
     client_id / domain_hash Client identity key (reserved exclusively for grouped split validation)
            future_* metrics                                     Post-decision window lookahead bias


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.